# Examen práctico: grupo 1, el MDP

Desarrollamos los entregables 1.1 y 1.2 sobre la representación del estado y la función de transición. Dejamos pendientes los entregables 1.3, 1.4 y las preguntas de integración.

Construimos ejemplos ilustrativos para analizar los supuestos del modelo y los distinguimos de las métricas de producción proporcionadas.


## Modelo que analizamos

Conservamos la transición original y representamos el estado como $(I_t,E_t,L_t)$: inventario, días hasta el vencimiento y categoría de demanda promedio de siete días. Interpretamos la acción como unidades pedidas y observamos una reposición inmediata, con inventario final limitado entre 0 y 100. En esta etapa analizamos sus supuestos sin implementar todavía un MDP corregido.


In [1]:
def transition(state, action):
    """Calcula el estado siguiente de forma determinista; la acción son unidades y la categoría de demanda permanece fija."""
    inventory, days_to_expiry, demand_level = state
    demand_map = {'bajo': 5, 'medio': 15, 'alto': 25, 'crítico': 40}
    daily_demand = demand_map[demand_level]
    new_inventory = min(100, max(0, inventory + action - daily_demand))
    new_days = max(1, days_to_expiry - 1)
    new_demand = demand_level
    return (new_inventory, new_days, new_demand)


resultados_produccion = {
    'stockouts_por_semana': 23,
    'productos_vencidos_por_semana': 41,
    'costo_almacenamiento_semanal': 8400,
    'costo_objetivo_semanal': 3200,
    'satisfaccion_cliente': 0.61,
}


## Entregable 1.1: propiedad de Markov

Usamos $H_t=(S_0,A_0,\ldots,S_t)$ para representar la historia observable. Exigimos que, al conocer el estado actual y la acción, la historia no aporte información adicional para predecir el siguiente estado:

$$P(S_{t+1}=s'\mid H_t,A_t=a)=P(S_{t+1}=s'\mid S_t,A_t=a).$$

**En el simulador.** Observamos que `transition` depende únicamente del estado actual y la acción. Si representamos esa función mediante $f$, obtenemos $P_{sim}(s'\mid s,a)=\mathbf{1}\{s'=f(s,a)\}$. Concluimos que la transición implementada sí cumple Markov; no confundimos una demanda poco realista con un incumplimiento de esta propiedad. Limitamos la conclusión a la transición disponible, porque no contamos con la implementación completa del entorno.

**En la farmacia real.** Identificamos información ausente que podría modificar la predicción del estado siguiente:

* No observamos pedidos pendientes, cantidades ni fechas de llegada; podemos tener el mismo inventario actual y distintas entregas al día siguiente.
* No conservamos la secuencia de demandas de los últimos siete días, su orden ni la media exacta al representar la demanda mediante una categoría.
* No incorporamos calendario, tendencias ni eventos que puedan alterar la distribución de demanda futura.
* No distinguimos cantidades y vencimientos por lote, por lo que no podemos determinar cuántas unidades dejarían de ser utilizables mañana.

Planteamos estas ausencias como mecanismos posibles del dominio, sin afirmar que hayan ocurrido en los registros de producción. Tampoco consideramos la criticidad clínica, por sí sola, una prueba de que la transición incumpla Markov.


### Contraejemplo con dos historias de pedidos

Construimos dos historias con el mismo estado observado $s=(20,14,\text{medio})$ y la misma acción $a=0$. En la primera suponemos una entrega pendiente de 30 unidades antes de las ventas de mañana; en la segunda suponemos que no existe esa entrega. Fijamos la demanda en 15 unidades para aislar el efecto de la información omitida.


In [2]:
estado_compartido = (20, 14, 'medio')
accion_compartida = 0
inventario_con_pedido_previo = min(100, max(0, 20 + 30 - 15))
inventario_sin_pedido_previo = min(100, max(0, 20 + 0 - 15))
print('Ejemplo hipotético: mismo estado y acción, distintas historias')
print(f'Historia con entrega pendiente: inventario siguiente = {inventario_con_pedido_previo}')
print(f'Historia sin entrega pendiente: inventario siguiente = {inventario_sin_pedido_previo}')
print(f'Transición original: {transition(estado_compartido, accion_compartida)}')
assert inventario_con_pedido_previo == 35
assert inventario_sin_pedido_previo == 5


Ejemplo hipotético: mismo estado y acción, distintas historias
Historia con entrega pendiente: inventario siguiente = 35
Historia sin entrega pendiente: inventario siguiente = 5
Transición original: (5, 13, 'medio')


Obtenemos 35 unidades con la entrega pendiente y 5 sin ella. Expresamos la diferencia como $P(I_{t+1}=35\mid H_t^{(1)},a=0)=1$ y $P(I_{t+1}=35\mid H_t^{(2)},a=0)=0$, aunque terminamos ambas historias en el mismo estado observado. Si admitimos ese funcionamiento real, no podemos representar ambas probabilidades mediante una transición independiente de la historia. Demostramos insuficiencia bajo ese supuesto, sin afirmar que existan pedidos pendientes en los datos entregados.

### Pérdida de información de la demanda

Interpretamos el promedio de siete días como una media móvil y escribimos su actualización como $m_{t+1}=m_t+(d_{t+1}-d_{t-6})/7$. No contamos en la tupla con la demanda más antigua $d_{t-6}$ que debemos retirar. Para mostrar la pérdida de información, construimos dos ventanas ordenadas de más antigua a más reciente, con igual media actual e igual demanda de mañana.


In [3]:
ventana_a = [25, 10, 10, 15, 15, 15, 15]
ventana_b = [10, 10, 15, 15, 15, 15, 25]
demanda_manana = 15
media_actual_a = sum(ventana_a) / 7
media_actual_b = sum(ventana_b) / 7
media_siguiente_a = (sum(ventana_a[1:]) + demanda_manana) / 7
media_siguiente_b = (sum(ventana_b[1:]) + demanda_manana) / 7
print(f'Medias actuales: {media_actual_a:.2f} y {media_actual_b:.2f}')
print(f'Medias siguientes: {media_siguiente_a:.2f} y {media_siguiente_b:.2f}')
assert media_actual_a == media_actual_b == 15
assert media_siguiente_a != media_siguiente_b


Medias actuales: 15.00 y 15.00
Medias siguientes: 13.57 y 15.71


Obtenemos medias siguientes de 13.57 y 15.71 a partir de una misma media actual de 15. Mostramos así que no podemos reconstruir la media siguiente con la información conservada. Como no disponemos de umbrales de clasificación, no afirmamos que ambas medias deban corresponder a categorías distintas.

**Consecuencia sobre el aprendizaje.** Llamamos $Z_t$ a la información omitida. Si suponemos que un estado ampliado $(S_t,Z_t)$ es suficiente, expresamos la transición observada como:

$$P(s'\mid H_t,a)=\sum_z P(s'\mid s,z,a)P(z\mid H_t,a).$$

Si cambiamos las probabilidades de la información oculta según la historia y esa información modifica las transiciones, obtenemos futuros distintos para un mismo par $(s,a)$. Al utilizar una única entrada $Q(s,a)$, agrupamos situaciones que pueden requerir decisiones diferentes; con una política que solo consulta $s$, no podemos distinguirlas.

Por ello no podemos aplicar directamente las garantías de un MDP estacionario a una representación real cuya suficiencia no hemos demostrado. No concluimos que el aprendizaje deba divergir: podemos obtener un compromiso o aprender correctamente el simulador y fallar al transferir la política a producción.


## Entregable 1.2: análisis de la función de transición

Extraemos las siguientes ecuaciones de la transición:

$$I_{t+1}=\min(100,\max(0,I_t+a_t-d(L_t))).$$

$$E_{t+1}=\max(1,E_t-1),\qquad L_{t+1}=L_t.$$

Usamos las demandas definidas en el modelo: $d(\text{bajo})=5$, $d(\text{medio})=15$, $d(\text{alto})=25$ y $d(\text{crítico})=40$. Por inducción obtenemos $L_t=L_0$ para cualquier secuencia de acciones. También observamos que, dentro de cada categoría, mantenemos una demanda diaria determinista con varianza condicional cero.

Concluimos que con este supuesto no representamos fluctuaciones diarias, estacionalidad, emergencias, tendencias ni caídas de rotación. Tampoco actualizamos la media de siete días. Identificamos cuatro subconjuntos sin transiciones entre categorías: si comenzamos en `bajo`, nunca llegamos a `crítico` dentro de esa trayectoria. Como no conocemos la inicialización del entorno, no descartamos que otros episodios comiencen en `crítico`.


In [4]:
estado = (60, 14, 'bajo')
trayectoria_original = [estado]
for _ in range(4):
    estado = transition(estado, 0)
    trayectoria_original.append(estado)

print('Transición original con acción de cero unidades')
print('Día | Inventario | Días al vencimiento | Demanda')
for dia, (inventario, vencimiento, nivel) in enumerate(trayectoria_original):
    print(f'{dia} | {inventario} | {vencimiento} | {nivel}')
assert all(s[2] == 'bajo' for s in trayectoria_original)


Transición original con acción de cero unidades
Día | Inventario | Días al vencimiento | Demanda
0 | 60 | 14 | bajo
1 | 55 | 13 | bajo
2 | 50 | 12 | bajo
3 | 45 | 11 | bajo
4 | 40 | 10 | bajo


### Sensibilidad a demandas que el modelo no contempla

Calculamos balances para secuencias ilustrativas y conservamos el recorte del inventario original. Mantenemos idénticos el inventario inicial y las acciones dentro de cada comparación, variamos únicamente la demanda y registramos las unidades no atendidas. Utilizamos esta función como herramienta de diagnóstico, sin definir todavía un nuevo estado ni una transición corregida para el agente.


In [5]:
def balance_diagnostico(inventario_inicial, acciones, demandas):
    """Compara demandas hipotéticas en el balance del MDP y registra unidades no atendidas que el recorte a cero oculta."""
    inventario = inventario_inicial
    filas = []
    for dia, (accion, demanda) in enumerate(zip(acciones, demandas), start=1):
        disponible = inventario + accion
        faltante = max(0, demanda - disponible)
        inventario = min(100, max(0, disponible - demanda))
        filas.append((dia, demanda, inventario, faltante))
    return filas


casos = [
    ('Demanda fija', 60, [0] * 4, [5, 5, 5, 5]),
    ('Pico hipotético', 60, [0] * 4, [5, 40, 40, 5]),
    ('Rotación fija', 50, [10] * 4, [5, 5, 5, 5]),
    ('Caída hipotética', 50, [10] * 4, [0, 0, 0, 0]),
]
balances = {}
print('Escenario | Demanda por día | Inventarios al cierre | Unidades no atendidas')
for nombre, inicial, acciones, demandas in casos:
    filas = balance_diagnostico(inicial, acciones, demandas)
    balances[nombre] = filas
    cierres = [fila[2] for fila in filas]
    no_atendidas = sum(fila[3] for fila in filas)
    print(f'{nombre} | {demandas} | {cierres} | {no_atendidas}')
assert [fila[2] for fila in balances['Demanda fija']] == [s[0] for s in trayectoria_original[1:]]
assert sum(fila[3] for fila in balances['Pico hipotético']) == 30
assert balances['Rotación fija'][-1][2] == 70
assert balances['Caída hipotética'][-1][2] == 90


Escenario | Demanda por día | Inventarios al cierre | Unidades no atendidas
Demanda fija | [5, 5, 5, 5] | [55, 50, 45, 40] | 0
Pico hipotético | [5, 40, 40, 5] | [55, 15, 0, 0] | 30
Rotación fija | [5, 5, 5, 5] | [55, 60, 65, 70] | 0
Caída hipotética | [0, 0, 0, 0] | [60, 70, 80, 90] | 0


Observamos que con demanda fija terminamos con 40 unidades y ningún faltante, mientras que con el pico terminamos sin inventario y acumulamos 30 unidades no atendidas. En la segunda comparación obtenemos 70 unidades finales con rotación fija y 90 cuando la demanda cae a cero. Mostramos así que podemos ocultar faltantes o acumulación al mantener una demanda invariable. No presentamos estas acciones ilustrativas como una ejecución de la política entrenada.

### Relación con los resultados de producción

* Relacionamos los **23 stockouts semanales** con la posibilidad de picos o cambios de demanda ausentes del modelo. Identificamos un mecanismo compatible, sin estimar cuántos incidentes explica ni equiparar unidades no atendidas con stockouts.
* Relacionamos el costo de almacenamiento de **8400 frente a 3200** con posibles acumulaciones cuando cae la rotación. Calculamos una razón de **2.625 veces el objetivo**, sin atribuir toda la diferencia al supuesto de demanda fija.
* Relacionamos los **41 productos vencidos por semana** con la posible permanencia de productos al caer la demanda. Además, observamos que detenemos el contador de vencimiento en 1 y no descontamos del inventario unidades vencidas en la transición.
* Reportamos la satisfacción de **0.61** como una métrica proporcionada. Al no conocer su definición ni contar con registros de servicio, no la interpretamos automáticamente como un porcentaje de clientes satisfechos ni establecemos una relación causal con los faltantes.

Distinguimos otras simplificaciones del estado y la transición: aplicamos la reposición sin demora, no registramos ventas perdidas al recortar el inventario a cero y no distinguimos lotes con diferentes vencimientos. Señalamos estas limitaciones dentro de nuestro componente, sin modificar los de otros grupos.

**Conclusión.** Sustentamos el argumento en la demanda invariable de la transición, en las diferencias de los balances al variar únicamente la demanda y en la compatibilidad de esos mecanismos con los síntomas reportados. Necesitaríamos series diarias de demanda, inventario, pedidos y vencimientos para confirmar su contribución causal. Con la evidencia disponible identificamos una limitación estructural, pero no podemos repartir los incidentes entre causas.


In [6]:
razon_costo = (
    resultados_produccion['costo_almacenamiento_semanal']
    / resultados_produccion['costo_objetivo_semanal']
)
print(f'Costo de almacenamiento / objetivo: {razon_costo:.3f}')
print('Verificamos los ejemplos de los entregables 1.1 y 1.2.')


Costo de almacenamiento / objetivo: 2.625
Verificamos los ejemplos de los entregables 1.1 y 1.2.


## Prompt de apoyo

> Queremos que nos ayudes a entender los entregables 1.1 y 1.2. Analiza el código Python y explícanos qué contiene para posteriormente completar estos entregables. No queremos que nos resuelvas todo; queremos que nos ayudes a plantear las fórmulas y los diferentes cambios que tendríamos que hacer para que todo funcione correctamente.

**Propósito del prompt:** buscamos comprender la representación del estado y la función de transición antes de completar los entregables. Delimitamos el apoyo al planteamiento de fórmulas y al análisis de los cambios necesarios para poder justificar nuestras decisiones.


## Referencias de apoyo

- GeeksforGeeks. (2025a, July 23). How to calculate moving averages in Python? GeeksforGeeks. https://www.geeksforgeeks.org/python/how-to-calculate-moving-averages-in-python/ 
- GeeksforGeeks. (2025b, July 31). Markov Chain. GeeksforGeeks. https://www.geeksforgeeks.org/machine-learning/markov-chain/ 
- GeeksforGeeks. (2026, July 8). Markov decision process. GeeksforGeeks. https://www.geeksforgeeks.org/machine-learning/markov-decision-process/
